In [0]:
import sys
import os
sys.path.append(os.path.abspath('..'))

from utils.utils_merge_into_tables import upsert_data
from utils.utils_transform_data import cast_columns, standardize_column_names, standardize_string_values
from pyspark.sql import functions as sf 
from pyspark.sql.dataframe import DataFrame


In [0]:
catalog = dbutils.widgets.get("catalog")
schema_bronze = dbutils.widgets.get("schema_bronze")
schema_silver = dbutils.widgets.get("schema_silver")
table_bronze = dbutils.widgets.get("table_bronze")
table_silver = dbutils.widgets.get("table_silver")
primary_keys = dbutils.widgets.get("primary_key")

In [0]:


df_bronze = spark.read.table(f"{catalog}.{schema_bronze}.{table_bronze}")

In [0]:
df_bronze = df_bronze.select("product_id", "product_category_name", "product_name_lenght", "product_description_lenght", "product_photos_qty", "product_weight_g", "product_length_cm", "product_height_cm", "product_width_cm")


data_type_mapping = {
    "product_id": "string",
    "product_category_name": "string",
    "product_name_lenght": "integer",
    "product_description_lenght": "integer",
    "product_photos_qty": "integer",
    "product_weight_g": "double",
    "product_length_cm": "double",
    "product_height_cm": "double",
    "product_width_cm": "double"
}

In [0]:
df_silver = (
    df_bronze
    .transform(lambda df: cast_columns(df, data_type_mapping))
    .transform(standardize_column_names)
    .transform(standardize_string_values)
    .withColumn("silver_update_date", sf.current_timestamp())
)


In [0]:
upsert_data(df_silver, table_silver,primary_keys)